---
title: "DRG Cleaning (Python) v2"

author: "Carlos Resurreccion"

date: "2024-11-19"

---


In [1]:
# Initialize variables
thread_offset = 0
sample_size_divisor = 625

# Whether to sample each split_part by sample_size_divisor
# (useful when iterating through code runs in quick succession)
to_sample = False  # Flag to indicate sampling
to_write = True    # Flag to enable writing outputs
to_flush = False   # Flag to enable flushing buffers
to_parallel = True # Flag for enabling parallel processing
to_debug = False   # Flag for enabling debugging

# Display the parallelization status
print("Parallelization:", to_parallel, "\n")

# Set verbose output based on debugging flag
verbose_output = True if to_debug else False

# Path to the year_to_load file
year_file_path = "/home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/cache/year_to_load.txt"

# Read the year from the file
with open(year_file_path, "r") as file:
    year_to_load = file.read().strip()  # .strip() removes any surrounding whitespace or newlines

# Create a suffix based on the `to_sample` flag and sample_size_divisor
if to_sample:
    suffix = f"_sampled_{sample_size_divisor}_"
else:
    suffix = "_full_"

Parallelization: True 



In [2]:
import pandas as pd
import os
import numpy as np
from grouper import seeker
from multiprocessing import Pool, cpu_count
import traceback
import swifter
import traceback
import sys
import io
import pyarrow

['scripts', 'tests', '.ipynb_checkpoints', 'drg-seeker.ipynb', '01-drg-cleaning-v2.ipynb', '02b-drg-grouping-py-v2.ipynb', 'requirements.txt', 'debug', 'execution.log', 'drg-cleaning-auto.ipynb', 'py_scripts', '00-drg-partial.ipynb', 'drg-cleaning-all-years.r', 'drg-spc-v2.ipynb', 'cache', 'libraries', 'drg-grouping-all-years.r', 'grouper', 'data', 'misc', 'r_scripts_v2', 'old_files', '02-drg-grouping-v2.ipynb']


In [3]:
# Construct the file path for the Feather file
feather_file_path = f"~/drg-pipeline/data-cleaning/data/checkpoints/checkpoint_7_py_input/python_input_{year_to_load}{suffix}.feather"

# Expand the `~` to the user's home directory
feather_file_path = os.path.expanduser(feather_file_path)

# Read the Feather file
pandas_df = pd.read_feather(feather_file_path)

# Print the DataFrame or process it as needed
print(pandas_df)

             id_series             date_adm             date_dis  patage  \
0        0000031755950  2019-01-11 20:45:00  2019-01-16 11:00:00    79.0   
1        0000048143200  2019-06-16 15:45:00  2019-06-20 08:15:00     1.0   
2        0000047188173  2019-08-09 20:21:00  2019-08-17 18:00:00    16.0   
3        0000026299297  2019-03-29 05:27:00  2019-04-05 11:59:00    75.0   
4        0000039560583  2019-05-30 18:30:00  2019-06-01 09:20:00    29.0   
...                ...                  ...                  ...     ...   
8679485  0000003363754  2019-12-05 01:27:00  2019-12-06 18:24:00     0.0   
8679486  0000005116351  2019-09-15 07:26:00  2019-09-17 19:44:00     0.0   
8679487  0000044622445  2019-09-21 17:35:00  2019-09-23 15:39:00     0.0   
8679488  0000001809346  2019-11-17 01:48:00  2019-11-19 14:01:00     0.0   
8679489  0000045993730  2019-11-17 13:00:00  2022-11-18 14:54:00    28.0   

        patsex  discharge   pdx  sdx1  sdx2  sdx3  ... proc13 proc14 proc15  \
0       

In [4]:
# Convert the column types explicitly
print("Converting data types")
pandas_df['patage'] = pd.to_numeric(pandas_df['patage'], errors='coerce')
pandas_df['birthweight'] = pd.to_numeric(pandas_df['birthweight'], errors='coerce')
pandas_df['discharge'] = pandas_df['discharge'].astype('Int64')

# Convert string columns to 'string' dtype and replace NA values with None
string_columns = ['id_series', 'patsex', 'pdx', 'sdx1', 'sdx2', 'sdx3', 'sdx4', 'sdx5', 'sdx6', 'sdx7', 'sdx8', 'sdx9', 'sdx10', 'sdx11', 'sdx12',
                  'proc1', 'proc2', 'proc3', 'proc4', 'proc5', 'proc6', 'proc7', 'proc8', 'proc9', 'proc10', 'proc11', 'proc12',
                  'proc13', 'proc14', 'proc15', 'proc16', 'proc17', 'proc18', 'proc19', 'proc20', 'date_adm', 'date_dis']

print("Replacing with None")
# Replace NA, pd.NA, '<NA>', 'None', 'NA' in string columns with None
# pandas_df[string_columns] = pandas_df[string_columns].replace([pd.NA, np.nan, '<NA>', 'None', 'NA'], None)
pandas_df = pandas_df.replace([pd.NA, np.nan, '<NA>', 'None', 'NA', -2147483648], None)
# pandas_df = pandas_df.replace(pd.NA, None)
# pandas_df = pandas_df.replace(np.nan, None)
# pandas_df = pandas_df.replace('<NA>', None)
# pandas_df = pandas_df.replace('None', None)
# pandas_df = pandas_df.replace('NA', None)
# pandas_df = pandas_df.replace(-2147483648, None)

print("Converting to string")
# Convert columns to string dtype after replacing the values
pandas_df[string_columns] = pandas_df[string_columns].astype('string')

print("Replacing -2147483648 with None")
# Replace -2147483648 with None in numeric columns, if necessary TODO: why do we do this a second time?
pandas_df = pandas_df.replace(-2147483648, None)

print("Filter discharge")
# Filter rows where 'discharge' is not in [1, 2, 3, 4, 9]
not_in_list_values = pandas_df.loc[~pandas_df['discharge'].isin([1, 2, 3, 4, 9]), 'discharge']

# Get unique values and their counts
unique_not_in_list_values = not_in_list_values.value_counts()

# Print the unique values and their counts
print(unique_not_in_list_values)

print("Generating info()")
pandas_df.info()

Converting data types
Replacing with None
Converting to string
Replacing -2147483648 with None
Filter discharge
Series([], Name: count, dtype: int64)
Generating info()
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8679490 entries, 0 to 8679489
Data columns (total 41 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   id_series    string
 1   date_adm     string
 2   date_dis     string
 3   patage       object
 4   patsex       string
 5   discharge    object
 6   pdx          string
 7   sdx1         string
 8   sdx2         string
 9   sdx3         string
 10  sdx4         string
 11  sdx5         string
 12  sdx6         string
 13  sdx7         string
 14  sdx8         string
 15  sdx9         string
 16  sdx10        string
 17  sdx11        string
 18  sdx12        string
 19  proc1        string
 20  proc2        string
 21  proc3        string
 22  proc4        string
 23  proc5        string
 24  proc6        string
 25  proc7        string
 26  proc8        s

In [5]:
# Sample 100,000 rows from the DataFrame
# full_df = pandas_df
# pandas_df = pandas_df.sample(n=100000, random_state=42)

In [6]:
# SINGLE-THREADED VERSION
# 
# print("Initializing Libraries")
# # Initialize the necessary libraries
# libs = seeker.Libraries()

# # Define a function to instantiate a Patient object for each row
# def process_patient(row, libs):
#     try:
#         # Convert the row to a dictionary and create a Patient object
#         patient = seeker.Patient(row.to_dict(), libs)
        
#         # Extract relevant attributes from the Patient object
#         result = {
#             'mdc': patient.mdc,
#             'pdc': patient.pdc,
#             'pccl': patient.pccl,
#             'drg': patient.drg,
#             'error_code': patient.error_code,
#             'warning_code': patient.warning_code
#         }
        
#         return pd.Series(result)
    
#     except Exception as e:
#         # Log the error and row information for debugging
#         print(f'''Error processing patient with id_series {row['id_series']}: {e}''')
        
#         # Optionally, you can log more information such as row content or traceback
#         traceback.print_exc()  # Print the full stack trace for more details
        
#         # Return None or default values for the error case
#         return pd.Series({
#             'mdc': None,
#             'pdc': None,
#             'pccl': None,
#             'drg': None,
#             'error_code': None,
#             'warning_code': None
#         })

# print("swifter.apply process_patient")
# # Apply the Patient class directly to each row using swifter
# pandas_df[['mdc', 'pdc', 'pccl', 'drg', 'error_code', 'warning_code']] = pandas_df.swifter.apply(
#     lambda row: process_patient(row, libs),
#     axis=1
# )
# print("Renaming columns")
# # Store the result in output to be retrieved by R
# output = pandas_df.rename(columns={'drg': 'py_drg'})
# print("Reordering columns")
# # Define the desired column order
# desired_columns = [
#     'id_series', 'mdc', 'pdc', 
#     'pccl', 'py_drg', 'error_code', 
#     'warning_code'
# ]
# print("Subsetting columns")
# # Reorder the DataFrame and drop any columns not in the desired list
# output = output[desired_columns]

# # # Define the renaming mapping
# # rename_mapping = {
# #     'patage': 'pat_age',
# #     'patsex': 'pat_sex',
# #     'birthweight': 'pat_bwt',
# #     'discharge': 'clin_discharge',
# #     'icd9_list': 'clin_rvs'
# # }

# # # Rename the columns
# # output = output.rename(columns=rename_mapping)

In [7]:
# Define a function to instantiate a Patient object for each row
def process_patient(row, libs):
    try:
        # Convert the row to a dictionary and create a Patient object
        patient = seeker.Patient(row.to_dict(), libs)

        # Extract relevant attributes from the Patient object
        result = {
            'mdc': patient.mdc,
            'pdc': patient.pdc,
            'pccl': patient.pccl,
            'drg': patient.drg,
            'error_code': patient.error_code,
            'warning_code': patient.warning_code
        }

        return result

    except Exception as e:
        # Log the error and row information for debugging
        print(f"Error processing patient with id_series {row['id_series']}: {e}")
        traceback.print_exc()

        # Return default values for error cases
        return {
            'mdc': None,
            'pdc': None,
            'pccl': None,
            'drg': None,
            'error_code': None,
            'warning_code': None
        }

# Function to process a chunk of the DataFrame
def process_chunk(chunk, libs):
    return chunk.apply(lambda row: pd.Series(process_patient(row, libs)), axis=1)

# Initialize the necessary libraries
print("Initializing Libraries")
libs = seeker.Libraries()

# Split DataFrame into chunks for multiprocessing
num_cores = cpu_count()  # Automatically detect the number of CPU cores
chunks = np.array_split(pandas_df, num_cores)  # Split the DataFrame into chunks

print(f"Processing using {num_cores} cores")

# Use multiprocessing to process each chunk in parallel
with Pool(num_cores) as pool:
    results = pool.starmap(process_chunk, [(chunk, libs) for chunk in chunks])

# Combine the results back into a single DataFrame
processed_df = pd.concat(results)

# Add the processed columns to the original DataFrame
pandas_df[['mdc', 'pdc', 'pccl', 'drg', 'error_code', 'warning_code']] = processed_df

# Rename and reorder columns
output = pandas_df.rename(columns={'drg': 'py_drg'})
desired_columns = [
    'id_series', 'mdc', 'pdc',
    'pccl', 'py_drg', 'error_code',
    'warning_code'
]
output = output[desired_columns]

Initializing Libraries
Processing using 24 cores
['J189']
['I500']
['J189', 'I500']
['I471']
['J189']
['J159']
['I500']
['I48']
['J189']
['I509']
['I509']
['I500']
['J960']
['R570']
['J09', 'J960']

['J189']['J189', 'I500']
['I501']
['I48', 'I110']
['J189']
['I500']
['I110']
['J189']
['I500']
['I619', 'J189']
['I501']
['J189', 'I110']
['J189']
['I500']
['J189']
['J189']
['J189']
['I48']
['I518']
['I518']
['I469']
['R570']
['I110']
['I38']
['J189']
['J189']
['I500']
['J189', 'J960']
['I48']
['I639']
['J189']
['I472', 'J189', 'I500', 'J960']
['I518', 'J189']
['J189']
['J189', 'I500']
['I469']['J189']

['I48']
['I48']
['I48']
['J960', 'N179']
['J960']
['I500']
['I501']
['J189']
['J189', 'I500']
['J189', 'I500']
['I509']
['J189']
['I469', 'J189']
['R570']


['J189']
['I110']['J189']['I500']
['I469']
['L89']
['J189']
['J690']
['I500']
['J189']
['J189']
['I500']
['I500']
['I500', 'R570']
['J189']
['I48']
['I48']
['I500']
['J189']
['J189']
['I48']
['I500']
['G467', 'J189']
['I509']
['I110']
[

In [8]:
output_dt = output
# Construct the file path
file_path = f"/home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/data/checkpoints/checkpoint_8_py_output/python_output_{year_to_load}{suffix}.feather"
# Save the DataFrame as a Feather file
output_dt.to_feather(file_path)